In [25]:
# auto reload modules
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cns.data_utils import main_load_data, load_samples_out, load_cns_out
from cns.utils.assemblies import hg19
from cns.utils.selection import select_CNS_samples, sample_head

## Load source files

In [31]:
sample_sel = sample_head(load_samples_out("TRACERx_samples.tsv"), 5)
cns_sel = select_CNS_samples(load_cns_out("TRACERx_cns_imp.tsv", True), sample_sel)
# drop column "total_cn"
cns_sel = cns_sel.drop(columns="total_cn")

In [43]:
sample_sel

,sex,type,chrom_count,chrom_missing,cover_bases_aut,cover_bases_sex,cover_bases_tot,cover_frac_aut,cover_frac_sex,cover_frac_tot,...,ane_major_cn_frac_aut,ane_minor_cn_frac_aut,ane_total_cn_frac_aut,ane_major_cn_sex,ane_minor_cn_sex,ane_total_cn_sex,ane_major_cn_frac_sex,ane_minor_cn_frac_sex,ane_total_cn_frac_sex,dataset
sample_id,,,,,,,,,,,,,,,,,,,,,
CRUK0002_SU_T1-R1,xy,LUAD,22,['chrX' 'chrY'],2651332635,0,2651332635,0.920271,0.0,0.856463,...,0.100823,0.286484,0.349214,214644126,0,214644126,1.0,0.0,1.0,primary
CRUK0002_SU_T1-R2,xy,LUAD,22,['chrX' 'chrY'],2651332635,0,2651332635,0.920271,0.0,0.856463,...,0.037322,0.269064,0.306335,214644126,0,214644126,1.0,0.0,1.0,primary
CRUK0001_SU_T1-R1,xy,LUAD,22,['chrX' 'chrY'],2634618947,0,2634618947,0.914470,0.0,0.851064,...,0.944610,0.731432,0.923578,214644126,0,214644126,1.0,0.0,1.0,primary
CRUK0001_SU_T1-R2,xy,LUAD,22,['chrX' 'chrY'],2634618947,0,2634618947,0.914470,0.0,0.851064,...,0.937371,0.727209,0.916973,214644126,0,214644126,1.0,0.0,1.0,primary
CRUK0001_SU_T1-R3,xy,LUAD,22,['chrX' 'chrY'],2634618947,0,2634618947,0.914470,0.0,0.851064,...,0.980202,0.757631,0.968706,214644126,0,214644126,1.0,0.0,1.0,primary


In [59]:
cns_sel.tail(10)

,sample_id,chrom,start,end,major_cn,minor_cn,ane
751,CRUK0002_SU_T1-R2,chr7,128354936,159138663,1,1,False
752,CRUK0002_SU_T1-R2,chr8,0,592407,2,1,True
753,CRUK0002_SU_T1-R2,chr8,592407,142476582,1,1,False
754,CRUK0002_SU_T1-R2,chr8,142476582,142480657,1,0,True
755,CRUK0002_SU_T1-R2,chr8,142480657,146364022,1,1,False
756,CRUK0002_SU_T1-R2,chr9,0,134385636,1,0,True
757,CRUK0002_SU_T1-R2,chr9,134385636,134388029,2,0,True
758,CRUK0002_SU_T1-R2,chr9,134388029,141213431,1,0,True
759,CRUK0002_SU_T1-R2,chrX,0,155270560,0,0,True
760,CRUK0002_SU_T1-R2,chrY,0,59373566,0,0,True


In [44]:
def get_expected_ploidy(column, chrom, is_xy=False, assembly=hg19):
    if chrom == assembly.chr_x:
        if is_xy:
            if column == "minor_cn" or column == "hapY_cn":
                return 0
            else:
                return 1
        else:
            if column == "total_cn":
                return 2
            else:
                return 1
    elif chrom == assembly.chr_y:
        if is_xy:
            if column == "minor_cn" or column == "hapY_cn":
                return 0
            else:
                return 1
        else:
            return 0
    else:
        if column == "total_cn":
            return 2
        else:
            return 1

In [49]:
def get_ane_for_cols(cns_df, samples_df, columns):
	is_ane = [cns_df.apply(lambda x: get_expected_ploidy(col, x["chrom"], samples_df.loc[x["sample_id"]]["sex"] == "xy") != x[col], axis=1) for col in columns]
	return is_ane

In [67]:
def get_ane_for_samples(cns_df, samples_df, columns, any=True):
	is_ane = get_ane_for_cols(cns_df, samples_df, columns)
	cns_df["ane"] = np.any(is_ane, axis=0) if any else np.all(is_ane, axis=0)
	cns_df["length"] = cns_df["end"] - cns_df["start"]
	res = cns_df[cns_df["ane"]].groupby("sample_id")["length"].sum()
	return res

In [68]:
get_ane_for_samples(cns_sel, sample_sel, ["major_cn", "minor_cn"])

sample_id
CRUK0001_SU_T1-R1    2943533719
CRUK0001_SU_T1-R2    2915745666
CRUK0001_SU_T1-R3    3043214453
CRUK0002_SU_T1-R1    1275550017
CRUK0002_SU_T1-R2    1097279087
Name: length, dtype: int64